# Block 2: Local Structures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imatge-upc/aa2/blob/main/notebooks/aa2_2_3_training_generalization.ipynb)


## 2.3: Training and Generalization

Train a small CIFAR-10 classifier by following six steps: **know the data → fix the evaluation → overfit one batch → fit the training set → regularize → tune and verify**.

Our model is $f_\theta$. The population loss $\mathcal{L}(\theta)=\mathbb{E}_{(x,y)}[\ell(f_\theta(x),y)]$ describes performance on the data distribution. We can compute only an empirical loss on a finite split $S$:

$$\hat{\mathcal{L}}_S(\theta)=\frac{1}{|S|}\sum_{(x_i,y_i)\in S}\ell(f_\theta(x_i),y_i).$$

We will distinguish fitting the training examples from performing well on unseen examples.

**Execution:** Work through the notebook one section at a time with PyTorch, torchvision, NumPy, matplotlib and IPython. Read the explanations, execute code cells in order, and pause at the questions and plots before continuing. The accompanying archive is only **16.95 MB**; normal execution needs neither the full CIFAR-10 download nor pretrained weights. CPU is supported; CUDA or Apple MPS is selected when available. The default trains on 3,000 images; a GPU runtime is recommended. For a shorter CPU session, set `RUN_MODE = 'quick'` in the first code cell to train on 1,000 images. Both modes print progress during each fit and keep the same validation/test samples. The saved outputs use the larger, standard mode. The saved CPU run took **10.4 minutes** on this macbook pro; GPU runtime has not been measured here. Timings and library versions are printed. The six-trial search is deliberately small.

**Google Colab: select a GPU runtime before running.** Colab calls the execution environment a **runtime**.

1. Open **Runtime → Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU** (or another available GPU), then click **Save**.
3. Run **only the first code cell** using its **▶** button. Check that its current output reports **`Device: cuda`**.
4. Continue **one code cell at a time, from top to bottom**. Inspect each result and discuss the questions before starting the next experiment.

If you change the runtime type after starting, begin again with the first code cell and work forward in order. The notebook selects CUDA automatically when available; CPU execution remains supported. The saved example outputs were produced on CPU. [Colab GPU setup instructions](https://docs.pytorch.org/tutorials/beginner/colab.html#enabling-cuda).

**Local and Colab data loading:** Locally, keep the archive at `assets/datasets/cifar10_training_recipe_subset_v2.npz` and start the kernel from the repository root or `notebooks/`. In Colab, open the badge above and follow the same section-by-section workflow. When you reach the data-loading cell, it downloads that same archive from GitHub into `data/`, then reuses it for the rest of the runtime. Publish the archive and notebook together on `main` first. If you use a fork or another branch, change `GITHUB_REPO` and `GITHUB_REF` in the data cell. Both paths verify the same SHA-256 checksum and use identical examples.

**Discussion checkpoints:** Pause after the data and split checks (sections 1–2), the one-batch and baseline experiments (3–4), and each regularization comparison (5). In section 6, inspect the search and seed-repeat results, then state which configuration validation selected and why **before executing the final test cell**.

Later cells use models and histories created earlier; saved outputs do not recreate those Python objects. For a discussion without retraining, read the saved outputs. To execute later experiments, first run the cells that create their inputs.

**Before running each experiment:** predict what will happen to the training and validation curves. Small validation differences may change across seeds; an intervention is allowed to make things worse.


In [ ]:
import copy
import hashlib
import math
from pathlib import Path
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
import torchvision
from torchvision import transforms
from torchvision.datasets.utils import download_url
from IPython.display import Markdown, display

NOTEBOOK_START = time.perf_counter()
RUN_MODE = 'standard'  # 'standard': 3,000 training images; 'quick': 1,000.
if RUN_MODE not in ('standard', 'quick'):
    raise ValueError("RUN_MODE must be 'standard' or 'quick'.")
TRAIN_PER_CLASS = 300 if RUN_MODE == 'standard' else 100
VAL_PER_CLASS = 100
TEST_PER_CLASS = 200
SEED = 7
FINAL_SEEDS = (7, 17, 27)
BATCH_SIZE = 64
MAX_EPOCHS = 30
SEARCH_EPOCHS = 8
N_TRIALS = 6
PATIENCE = 5
torch.set_num_threads(min(4, torch.get_num_threads()))
device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize()
    elif device.type == 'mps':
        torch.mps.synchronize()


def show_table(headers, rows):
    def escaped(value):
        return str(value).replace('|', r'\|')
    lines = ['| ' + ' | '.join(map(escaped, headers)) + ' |',
             '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines += ['| ' + ' | '.join(map(escaped, row)) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))


seed_everything(SEED)
print(f'Device: {device}; CPU threads: {torch.get_num_threads()}')
print(f'Run mode: {RUN_MODE}; train {10 * TRAIN_PER_CLASS:,}; '
      f'validation {10 * VAL_PER_CLASS:,}; reserved test {10 * TEST_PER_CLASS:,}')
print(f'Python {platform.python_version()}, PyTorch {torch.__version__}, '
      f'torchvision {torchvision.__version__}, NumPy {np.__version__}, '
      f'matplotlib {plt.matplotlib.__version__}')
print('Seeds repeat runs on this setup; exact results can differ across devices and versions.')


**A toy experiment on a real dataset**

We deliberately fit only **3,000 images: 300 per class**, or **6% of [CIFAR-10's 50,000 original training images](https://www.cs.toronto.edu/~kriz/cifar.html)**. This gives the recipe more varied training examples while retaining a small teaching budget. The optional quick mode uses 1,000 of those training images (100 per class), with the same validation and test samples. Every experiment in a run uses the selected mode; do not change modes halfway through. The real images still contain substantial variation in pose, background and appearance.

| Teaching choice | Consequence for interpreting results |
| --- | --- |
| Very little training data | Many visual variations may be poorly represented. Augmented views reuse the same original scenes. |
| A simple CNN trained from scratch | The model must learn its features from these 3,000 images; it has no pretrained visual knowledge. |
| At most 30 epochs; six short search trials | Some settings may need more optimization or different hyperparameters. This is a limited exploration. |
| One small validation split and three training seeds | Small differences can be noisy. Repeated seeds do not measure uncertainty from choosing another split. |

Expect useful diagnostics and modest absolute accuracy. We will check mechanisms, measure validation changes, and verify the selected model on held-out data. **A technique's effect depends on its setting and the problem.** Limited data is a plausible contributor to weak results, but this experiment does not establish it as the sole cause. A training-size learning curve would be needed to investigate that explanation.


### 1. Know Your Data

CIFAR-10 contains 32 × 32 RGB images from ten classes. This archive contains a **4,000-image development pool** from the official training partition and **2,000 reserved test images** from the official test partition. The next section splits the development pool into 3,000 training and 1,000 validation images in standard mode. Quick mode uses a nested 1,000-image training subset, leaving the other 2,000 training candidates unused.

The export selects a fixed number per class and rejects duplicate pixel hashes. Original image indices are retained for provenance. We check split integrity here, but use no test predictions to make training decisions. Examples and visual inspection use only the development pool. This version uses a fresh test cohort: the 1,000 test images examined in the earlier small version, including exact pixel copies, were excluded during export. A new versioned filename prevents an old Colab cache from silently supplying the earlier data.


In [ ]:
SUBSET_NAME = 'cifar10_training_recipe_subset_v2.npz'
SUBSET_SHA256 = '0f2d4ba224b4a7c3aea490b3c767c32c6fd1afde48b775558ea975f33702936a'
GITHUB_REPO = 'imatge-upc/aa2'  # Change this if you use a fork.
GITHUB_REF = 'main'            # Branch, tag or commit containing the archive.
DATA_CACHE = Path('data')

# Local checkout: support kernels started at the repo root or in notebooks/.
# Colab: reuse a previous download in this runtime, or fetch the small archive.
candidates = [Path('assets/datasets') / SUBSET_NAME,
              Path('../assets/datasets') / SUBSET_NAME,
              DATA_CACHE / SUBSET_NAME]
subset_path = next((p for p in candidates if p.is_file()), None)
if subset_path is None:
    subset_url = (f'https://raw.githubusercontent.com/{GITHUB_REPO}/{GITHUB_REF}/'
                  f'assets/datasets/{SUBSET_NAME}')
    print(f'Downloading the teaching subset from {GITHUB_REPO} ({GITHUB_REF})...')
    try:
        download_url(subset_url, root=str(DATA_CACHE), filename=SUBSET_NAME)
    except (OSError, RuntimeError) as error:
        raise RuntimeError(
            f'Could not download {subset_url}. Publish assets/datasets/{SUBSET_NAME} '
            f'in {GITHUB_REPO} at {GITHUB_REF}, or update GITHUB_REPO/GITHUB_REF '
            'to the repository and revision containing your dataset.'
        ) from error
    subset_path = DATA_CACHE / SUBSET_NAME

if hashlib.sha256(subset_path.read_bytes()).hexdigest() != SUBSET_SHA256:
    raise ValueError(f'Archive checksum mismatch: {subset_path}. '
                     'Use the dataset version distributed with this notebook.')
with np.load(subset_path, allow_pickle=False) as archive:
    pool_images, pool_labels = archive['pool_images'], archive['pool_labels']
    pool_original_indices = archive['pool_indices']
    test_images, test_labels = archive['test_images'], archive['test_labels']
    test_original_indices = archive['test_indices']
    classes = archive['classes'].tolist()
    previous_test_indices = archive['previous_test_indices']
    assert int(archive['schema_version']) == 2
K = len(classes)
print(f'Dataset file: {subset_path}')
print(f'Development pool: {pool_images.shape}; reserved test: {test_images.shape}')
print(f'Archive: {subset_path.stat().st_size / 1e6:.2f} MB; SHA-256 verified.')


In [ ]:
fig, axes = plt.subplots(2, K, figsize=(14, 3.5))
for label, name in enumerate(classes):
    example_indices = np.flatnonzero(pool_labels == label)[:2]
    for row, index in enumerate(example_indices):
        axes[row, label].imshow(pool_images[index])
        axes[row, label].axis('off')
    axes[0, label].set_title(name, fontsize=9)
fig.suptitle('Development examples: which classes look difficult to separate?')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 2.5))
ax.bar(classes, np.bincount(pool_labels, minlength=K), color='tab:blue')
ax.set(ylabel='Images', title='Development-pool class balance')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


An exact duplicate can make evaluation too optimistic if one copy is used for training and another for validation or testing. Hash raw pixels before normalization or augmentation. This detects identical images; it does not detect every near-duplicate or related scene.


In [ ]:
def pixel_hashes(images):
    return [hashlib.sha256(image.tobytes()).digest() for image in images]


pool_hashes = pixel_hashes(pool_images)
test_hashes = pixel_hashes(test_images)
pool_duplicates = len(pool_hashes) - len(set(pool_hashes))
test_duplicates = len(test_hashes) - len(set(test_hashes))
pool_test_overlap = len(set(pool_hashes) & set(test_hashes))
show_table(['Check', 'Count'], [
    ['Duplicates inside development pool', pool_duplicates],
    ['Duplicates inside reserved test', test_duplicates],
    ['Identical images crossing development/test', pool_test_overlap],
    ['Official-test indices reused from the previous version',
     len(set(test_original_indices) & set(previous_test_indices))],
])
assert pool_duplicates == test_duplicates == pool_test_overlap == 0
assert len(np.unique(pool_original_indices)) == len(pool_images)
assert len(np.unique(test_original_indices)) == len(test_images)
assert not (set(test_original_indices) & set(previous_test_indices))
print('Original indices belong to different official partitions; pixel hashes check cross-partition overlap.')


**Questions:** Which confusions do you expect from the examples? Would balanced classes alone guarantee an easy problem? What would happen if duplicate copies crossed a split boundary?


### 2. Fix the Evaluation

The official training/test boundary is preserved. Within the development pool, stratified sampling creates **300 training and 100 validation images per class** in standard mode. Quick mode takes the first 100 training candidates per class from that same seeded ordering. The validation set stays unchanged. The reserved official-test sample has **200 per class** in both modes. Keep these indices fixed for every experiment.

* **Training:** fit $\theta$ and estimate input-normalization statistics.
* **Validation:** select checkpoints, regularization and hyperparameters.
* **Test:** one final evaluation stage after all choices are locked.

Our primary reported metric is **accuracy**. We select checkpoints and configurations by **minimum validation cross-entropy**, with deterministic first-occurrence tie breaking. This uses confidence as well as correctness; we report both loss and accuracy throughout.

The trivial baseline always predicts the most common **training** class. With a tie, choose the smallest class ID. Uniform class probabilities give cross-entropy $-\log(1/K)=\log K\approx2.303$. A randomly initialized network should start nearby, although its logits need not be exactly equal.


In [ ]:
split_rng = np.random.default_rng(SEED)
train_indices, val_indices = [], []
for label in range(K):
    indices = split_rng.permutation(np.flatnonzero(pool_labels == label))
    # Reserve validation first, independently of the chosen training size.
    train_indices.extend(indices[:TRAIN_PER_CLASS])
    val_indices.extend(indices[300:300 + VAL_PER_CLASS])
# Shuffle validation first so even its ordering is identical across modes.
val_indices = split_rng.permutation(val_indices)
train_indices = split_rng.permutation(train_indices)
train_images, train_labels = pool_images[train_indices], pool_labels[train_indices]
val_images, val_labels = pool_images[val_indices], pool_labels[val_indices]
assert not (set(train_indices) & set(val_indices))
assert not (set(pixel_hashes(train_images)) & set(pixel_hashes(val_images)))
assert len(pool_images) == K * (300 + VAL_PER_CLASS)
assert len(train_indices) + len(val_indices) <= len(pool_images)
assert np.all(np.bincount(train_labels, minlength=K) == TRAIN_PER_CLASS)
assert np.all(np.bincount(val_labels, minlength=K) == VAL_PER_CLASS)
assert np.all(np.bincount(test_labels, minlength=K) == TEST_PER_CLASS)
majority_class = int(np.bincount(train_labels, minlength=K).argmax())
majority_val_accuracy = float(np.mean(val_labels == majority_class))
show_table(['Split', 'Images', 'Images/class'], [
    ['Train', len(train_labels), TRAIN_PER_CLASS],
    ['Validation', len(val_labels), VAL_PER_CLASS],
    ['Test (reserved)', len(test_labels), TEST_PER_CLASS],
])
print(f'Majority-class baseline: always {classes[majority_class]}; '
      f'validation accuracy {majority_val_accuracy:.1%}.')

# Estimate channel statistics using training pixels only.
pixels = torch.from_numpy(train_images).float() / 255
mean = pixels.mean(dim=(0, 1, 2)).tolist()
std = pixels.std(dim=(0, 1, 2), unbiased=False).tolist()
del pixels
plain_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(mean, std)])
augmented_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), plain_transform,
])


class CIFARSubset(Dataset):
    def __init__(self, images, labels, transform):
        self.images, self.labels, self.transform = images, labels, transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.transform(Image.fromarray(self.images[index])), int(self.labels[index])


train_plain = CIFARSubset(train_images, train_labels, plain_transform)
train_augmented = CIFARSubset(train_images, train_labels, augmented_transform)
val_data = CIFARSubset(val_images, val_labels, plain_transform)
train_eval_loader = DataLoader(train_plain, batch_size=128, shuffle=False, num_workers=0)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False, num_workers=0)
print('Training channel mean:', np.round(mean, 3), '; std:', np.round(std, 3))


Our CNN has three convolution/pooling stages followed by a dense head. It is intentionally large enough to fit this small training subset. The optional BatchNorm and Dropout layers let us change one ingredient without rewriting the network. Its output is **ten raw logits**; `CrossEntropyLoss` applies the needed log-softmax internally.


In [ ]:
def make_model(width=32, batch_norm=False, dropout=0.0):
    layers = []
    channels = 3
    for out_channels in (width, 2 * width, 4 * width):
        layers.append(nn.Conv2d(channels, out_channels, 3, padding=1))
        if batch_norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.extend([nn.ReLU(), nn.MaxPool2d(2)])
        channels = out_channels
    layers.extend([nn.Flatten(), nn.Linear(4 * width * 4 * 4, 8 * width),
                   nn.ReLU(), nn.Dropout(dropout), nn.Linear(8 * width, K)])
    return nn.Sequential(*layers)


criterion = nn.CrossEntropyLoss()
seed_everything(SEED)
initial_model = make_model().to(device)
initial_model.eval()
initial_loss_sum = torch.zeros((), device=device)
with torch.inference_mode():
    for images, labels in train_eval_loader:
        logits = initial_model(images.to(device))
        initial_loss_sum += nn.functional.cross_entropy(logits, labels.to(device), reduction='sum')
initial_loss = initial_loss_sum.item() / len(train_plain)
uniform_loss = criterion(torch.zeros(20, K), torch.arange(20) % K).item()
print(initial_model)
print(f'Parameters: {sum(p.numel() for p in initial_model.parameters()):,}')
print(f'Uniform logits: {uniform_loss:.4f}; log K: {math.log(K):.4f}; '
      f'initial empirical training loss: {initial_loss:.4f}')
assert math.isclose(uniform_loss, math.log(K), rel_tol=1e-6)


**Observe:** The random network and the majority-class predictor are different baselines. The first checks the scale of the loss; the second establishes a trivial accuracy to beat.

**Question:** If the initial loss were enormous, what would you inspect about logits, targets and preprocessing? Why should normalization statistics come from training images?


### 3. Overfit One Batch

Before the full experiment, repeatedly fit the **same 20 examples**, two from each class. Use no augmentation, weight decay, dropout or BatchNorm. The batch is small and fixed, so a sufficiently flexible model should drive $\hat{\mathcal{L}}$ close to zero.

**Predict before running:** What training accuracy should we reach? Would this result tell us anything about unseen images?


In [ ]:
mini_indices = np.concatenate([np.flatnonzero(train_labels == label)[:2] for label in range(K)])
mini_x = torch.stack([train_plain[int(i)][0] for i in mini_indices]).to(device)
mini_y = torch.tensor(train_labels[mini_indices], dtype=torch.long, device=device)
seed_everything(SEED)
mini_model = make_model().to(device)
mini_optimizer = torch.optim.AdamW(mini_model.parameters(), lr=3e-3, weight_decay=0.0)
mini_steps, mini_losses, mini_accuracies = [], [], []
synchronize()
mini_start = time.perf_counter()
for step in range(151):
    mini_model.eval()
    with torch.inference_mode():
        logits = mini_model(mini_x)
        measured_loss = criterion(logits, mini_y).item()
        measured_accuracy = (logits.argmax(1) == mini_y).float().mean().item()
    mini_steps.append(step)
    mini_losses.append(measured_loss)
    mini_accuracies.append(measured_accuracy)
    if measured_loss < 1e-3 or step == 150:
        break
    mini_model.train()
    mini_optimizer.zero_grad(set_to_none=True)
    criterion(mini_model(mini_x), mini_y).backward()
    mini_optimizer.step()
synchronize()
mini_seconds = time.perf_counter() - mini_start
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].semilogy(mini_steps, mini_losses)
axes[0].axhline(1e-3, color='gray', linestyle='--', label='Loss target')
axes[0].set(ylabel='Cross-entropy')
axes[0].legend()
axes[1].plot(mini_steps, mini_accuracies)
axes[1].set(ylabel='Accuracy', ylim=(0, 1.05))
for ax in axes:
    ax.set_xlabel('Parameter updates')
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()
print(f'{len(mini_y)} examples; {mini_steps[-1]} updates; loss {mini_losses[-1]:.6f}; '
      f'accuracy {mini_accuracies[-1]:.1%}; {mini_seconds:.1f} s')
assert mini_accuracies[-1] == 1.0 and mini_losses[-1] < 0.02, 'Investigate the one-batch sanity check.'


**Observe:** This verifies that the data, model, loss, gradient computation and optimizer can work together. Memorization is expected here.

If it fails, inspect the input/target pairs, target dtype and range, logits, nonzero gradients, `zero_grad → backward → step`, learning rate and accidental regularization. **Restart from a fresh model for the next step.**


### 4. Fit the Training Set

We now train on all 3,000 training images in standard mode (1,000 in quick mode). A shared `fit` function keeps comparisons consistent: each run creates a **fresh model and optimizer**, resets the seed, and uses a separate seeded generator for batch order. All optimizers are AdamW; the baseline explicitly uses `weight_decay=0.0`.

At the end of every epoch, both curves are measured on **clean images in evaluation mode**, without gradients. This matters for Dropout, BatchNorm and augmentation. Batch losses are weighted by the number of examples before averaging.

The best checkpoint minimizes validation loss. For the demonstrations we continue to the full budget to reveal overfitting. Later, `patience=5` will stop a fit after five epochs without improvement and restore its best checkpoint.

A large training/validation gap is easy to reveal with so few images. Fitting these examples perfectly can coexist with weak recognition of new ones. Inspect training loss for optimization problems and validation behavior for generalization; the absolute accuracy also depends on data coverage, architecture and the available training budget.


In [ ]:
@torch.inference_mode()
def evaluate(model, loader, predictions=False):
    model.eval()
    loss_sum = torch.zeros((), device=device)
    correct = torch.zeros((), device=device, dtype=torch.long)
    all_predictions = []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss_sum += nn.functional.cross_entropy(logits, labels, reduction='sum')
        predicted = logits.argmax(1)
        correct += (predicted == labels).sum()
        if predictions:
            all_predictions.append(predicted.cpu())
    result = {'loss': loss_sum.item() / len(loader.dataset),
              'acc': correct.item() / len(loader.dataset)}
    if predictions:
        result['predictions'] = torch.cat(all_predictions).numpy()
    return result


BASE_CONFIG = dict(width=32, batch_norm=False, dropout=0.0,
                   augment=False, lr=1e-3, weight_decay=0.0)


def fit(name, *, seed=SEED, epochs=MAX_EPOCHS, patience=None, **changes):
    config = {**BASE_CONFIG, **changes}
    seed_everything(seed)
    model = make_model(config['width'], config['batch_norm'], config['dropout']).to(device)
    training_data = train_augmented if config['augment'] else train_plain
    loader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
                        generator=torch.Generator().manual_seed(seed))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'],
                                 weight_decay=config['weight_decay'])
    history = {key: [] for key in ('train_loss', 'val_loss', 'train_acc', 'val_acc')}
    best_loss, best_epoch, best_state, bad_epochs = float('inf'), 0, None, 0
    synchronize()
    start = time.perf_counter()
    print(f'{name}: seed={seed}, at most {epochs} epochs', flush=True)
    for epoch in range(1, epochs + 1):
        model.train()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(images), labels)
            if not torch.isfinite(loss):
                raise RuntimeError(f'{name}: non-finite training loss; inspect the learning rate.')
            loss.backward()
            optimizer.step()
        train_metrics = evaluate(model, train_eval_loader)
        val_metrics = evaluate(model, val_loader)
        for key, value in zip(history, (train_metrics['loss'], val_metrics['loss'],
                                       train_metrics['acc'], val_metrics['acc'])):
            history[key].append(value)
        if val_metrics['loss'] < best_loss:
            best_loss, best_epoch, bad_epochs = val_metrics['loss'], epoch, 0
            best_state = {key: value.detach().cpu().clone()
                          for key, value in model.state_dict().items()}
        else:
            bad_epochs += 1
        if epoch == 1 or epoch % 5 == 0:
            print(f'  epoch {epoch:2d}: loss {train_metrics["loss"]:.3f}/{val_metrics["loss"]:.3f}; '
                  f'accuracy {train_metrics["acc"]:.1%}/{val_metrics["acc"]:.1%} (train/val)', flush=True)
        if patience is not None and bad_epochs >= patience:
            print(f'  Early stop at epoch {epoch}.', flush=True)
            break
    synchronize()
    seconds = time.perf_counter() - start
    model.load_state_dict(best_state)
    model.eval()
    k = best_epoch - 1
    selected = {key: values[k] for key, values in history.items()}
    print(f'  Selected epoch {best_epoch}: val loss {best_loss:.3f}, '
          f'val accuracy {selected["val_acc"]:.1%}; {seconds:.1f} s', flush=True)
    return dict(name=name, config=config, seed=seed, history=history, selected=selected,
                best_epoch=best_epoch, epochs_run=epoch, seconds=seconds, model=model.cpu())


def plot_runs(runs, title, limit=None):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for index, run in enumerate(runs):
        h = run['history']
        n = min(len(h['val_loss']), limit) if limit is not None else len(h['val_loss'])
        color = f'C{index}'
        for ax, metric in zip(axes, ('loss', 'acc')):
            ax.plot(range(1, n + 1), h[f'train_{metric}'][:n], color=color,
                    label=f'{run["name"]}: train')
            ax.plot(range(1, n + 1), h[f'val_{metric}'][:n], color=color, linestyle='--',
                    label=f'{run["name"]}: validation')
    axes[0].set_ylabel('Cross-entropy')
    axes[1].set(ylabel='Accuracy', ylim=(0, 1.02))
    for ax in axes:
        ax.set_xlabel('Epoch')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.suptitle(title + ' — solid: clean train; dashed: validation')
    plt.tight_layout()
    plt.show()


In [ ]:
baseline_run = fit('Unregularized CNN')
plot_runs([baseline_run], 'Fitting the full training subset')
last = {key: values[-1] for key, values in baseline_run['history'].items()}
show_table(['Checkpoint', 'Train loss', 'Val loss', 'Train accuracy', 'Val accuracy'], [
    ['Last epoch', f'{last["train_loss"]:.3f}', f'{last["val_loss"]:.3f}',
     f'{last["train_acc"]:.1%}', f'{last["val_acc"]:.1%}'],
    [f'Selected epoch {baseline_run["best_epoch"]}',
     f'{baseline_run["selected"]["train_loss"]:.3f}', f'{baseline_run["selected"]["val_loss"]:.3f}',
     f'{baseline_run["selected"]["train_acc"]:.1%}', f'{baseline_run["selected"]["val_acc"]:.1%}'],
])


#### How BatchNorm and LayerNorm Work

Before fitting another CNN, inspect a `4 × 3` tensor: **rows are examples, columns are features**. `BatchNorm1d(3)` computes statistics down each feature column; `LayerNorm(3)` computes statistics across each example's row. We disable learnable scale and shift to isolate normalization.

**Predict before running:** Along which axis should each output have mean approximately zero and variance approximately one? Will BatchNorm give the same output after switching to `eval()` following just one batch?


In [ ]:
norm_demo_x = torch.tensor([[1., 10., 100.],
                            [2., 20.,  80.],
                            [3., 30.,  60.],
                            [4., 40.,  40.]])
norm_demo_bn = nn.BatchNorm1d(3, affine=False)
norm_demo_ln = nn.LayerNorm(3, elementwise_affine=False)
norm_demo_bn.train()
norm_demo_bn_y = norm_demo_bn(norm_demo_x)
norm_demo_ln_y = norm_demo_ln(norm_demo_x)
print('Input (examples × features):\n', norm_demo_x)
for norm_demo_name, norm_demo_y, norm_demo_axis in [
    ('BatchNorm: per feature', norm_demo_bn_y, 0),
    ('LayerNorm: per example', norm_demo_ln_y, 1),
]:
    print(f'\n{norm_demo_name}:\n', norm_demo_y)
    print('Means:', norm_demo_y.mean(dim=norm_demo_axis).detach().numpy().round(3))
    print('Variances:', norm_demo_y.var(dim=norm_demo_axis, unbiased=False).detach().numpy().round(3))

norm_demo_running_mean = norm_demo_bn.running_mean.clone()
norm_demo_running_var = norm_demo_bn.running_var.clone()
print('\nBatchNorm running mean:', norm_demo_running_mean)
print('BatchNorm running variance:', norm_demo_running_var)
norm_demo_bn.eval()
norm_demo_ln.eval()
norm_demo_bn_eval = norm_demo_bn(norm_demo_x)
print('BatchNorm evaluation output:\n', norm_demo_bn_eval)
print('Running statistics unchanged by evaluation:',
      torch.equal(norm_demo_bn.running_mean, norm_demo_running_mean)
      and torch.equal(norm_demo_bn.running_var, norm_demo_running_var))
print('LayerNorm output unchanged by evaluation:',
      torch.allclose(norm_demo_ln(norm_demo_x), norm_demo_ln_y))


**Observe:** BatchNorm's evaluation output uses running statistics. After one batch these still mix the initial values with the observed statistics, so they are not the current batch's mean and variance. LayerNorm uses each example's own statistics in both modes. The small `eps` prevents exactly unit output variance; BatchNorm uses a biased variance for normalization and an unbiased estimate when updating its running variance.

For CNN features shaped `(B, C, H, W)`, the axes become:

| Layer | Values sharing each mean and variance | Example: `(8, 16, 4, 4)` |
| --- | --- | --- |
| `BatchNorm2d(C)` in training | `B × H × W` values within one channel | 128 values per channel; 16 statistic pairs |
| `LayerNorm((C, H, W))` | `C × H × W` values within one image | 256 values per image; 8 statistic pairs |

Our CNN enables BatchNorm's learnable scale and shift. Both layers preserve tensor shape. Normalization changes how optimization behaves; it does not supply missing training examples or guarantee higher validation accuracy. Next, compare that effect in actual short fits.


#### Diagnose Capacity, Learning Rate and BatchNorm

Compare three short runs with the **first six epochs** of the baseline: a narrower CNN, a much smaller learning rate, and BatchNorm inserted after each convolution. Each comparison changes one ingredient. Initialization and batch order use the same seed; differently shaped networks cannot have identical parameter tensors.

**Predict before running:** Which setting will learn most slowly? Does a six-epoch comparison tell us the best accuracy each architecture can eventually achieve?


In [ ]:
diagnostic_runs = [
    fit('Narrow CNN', epochs=6, width=8),
    fit('Small learning rate', epochs=6, lr=3e-5),
    fit('With BatchNorm', epochs=6, batch_norm=True),
]
plot_runs([baseline_run, *diagnostic_runs], 'Short training diagnostics', limit=6)
show_table(['Run', 'Epoch-6 train loss', 'Epoch-6 val accuracy', 'Parameters'], [
    [run['name'], f'{run["history"]["train_loss"][5]:.3f}',
     f'{run["history"]["val_acc"][5]:.1%}',
     f'{sum(p.numel() for p in run["model"].parameters()):,}']
    for run in [baseline_run, *diagnostic_runs]
])


**Observe:** Capacity and optimization interact: poor training performance may reflect too few updates or a poor learning rate. BatchNorm changes optimization and maintains running statistics; its evaluation curve uses those statistics, not the current validation batch.

**Questions:** Which settings reduce training loss fastest? Where does a generalization gap appear? Can validation loss rise while accuracy remains approximately constant?

These are short diagnostics, not a ranking of fully tuned architectures. For the regularization comparisons below, keep the original CNN and learning rate fixed, so only the named regularizer changes.


### 5. Regularize, One Technique at a Time

Compare three independent interventions against the same unregularized CNN. All use the same seed, batch order, learning rate and 30-epoch budget. We add **only the named technique**, resetting weights and optimizer each time. The next section chooses a family using validation results.

Each technique changes the learning problem or the preference among solutions. Augmentation reuses existing images under assumed label-preserving transformations; it does not introduce independent new scenes. Dropout and weight decay can reduce overfitting, but an unsuitable strength or a short training budget can also hurt. Look for a measured validation benefit rather than expecting every added ingredient to improve accuracy.

#### Augmentation

Random crops and horizontal flips supply new training views while retaining labels. Validation and clean training evaluation remain deterministic. **Predict:** how should augmentation affect memorization and the gap between curves?


In [ ]:
seed_everything(SEED)
view_transform = transforms.Compose([transforms.RandomCrop(32, padding=4),
                                     transforms.RandomHorizontalFlip()])
example = Image.fromarray(train_images[0])
fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for i, ax in enumerate(axes):
    ax.imshow(example if i == 0 else view_transform(example))
    ax.set_title('Original' if i == 0 else f'View {i}')
    ax.axis('off')
plt.tight_layout()
plt.show()
augmentation_run = fit('Augmentation only', augment=True)
plot_runs([baseline_run, augmentation_run], 'Change only augmentation')


#### Weight Decay with AdamW

AdamW applies decoupled shrinkage to the parameters. Here we use `weight_decay=0.1`, with no augmentation or dropout, and keep the learning rate fixed. Its effect depends on the learning rate and the number of updates; a tiny value may barely matter in a short run. Our compact example decays all parameters.

First inspect one numerical update with a zero data gradient. **Predict:** will a smaller parameter norm necessarily imply better validation accuracy?


In [ ]:
# Isolate AdamW's shrinkage using one parameter and a zero data gradient.
decay_demo_lr, decay_demo_strength = 0.1, 0.2
decay_demo_plain = nn.Parameter(torch.tensor([2.0]))
decay_demo_weight = nn.Parameter(torch.tensor([2.0]))
for decay_demo_parameter, decay_demo_wd in [
    (decay_demo_plain, 0.0), (decay_demo_weight, decay_demo_strength),
]:
    decay_demo_optimizer = torch.optim.AdamW([decay_demo_parameter],
                                            lr=decay_demo_lr, weight_decay=decay_demo_wd)
    # grad=None would skip this parameter's update; zero isolates weight decay.
    decay_demo_parameter.grad = torch.zeros_like(decay_demo_parameter)
    decay_demo_optimizer.step()
decay_demo_expected = 2.0 * (1 - decay_demo_lr * decay_demo_strength)
show_table(['Update', 'Before', 'After'], [
    ['AdamW, zero gradient, no decay', '2.0000', f'{decay_demo_plain.item():.4f}'],
    ['AdamW, zero gradient, with decay', '2.0000', f'{decay_demo_weight.item():.4f}'],
])
print(f'Expected shrinkage: 2 × (1 − {decay_demo_lr} × {decay_demo_strength}) = {decay_demo_expected:.4f}')


**Observe:** With a zero data gradient, the parameter stays at 2 without decay and shrinks to 1.96 with decay: $\theta\leftarrow(1-\eta\lambda)\theta$. AdamW applies this shrinkage separately from its adaptive gradient update. The toy learning rate and decay above are deliberately large enough to make one update visible; the CNN experiment below uses its original settings.

Shrinkage is a mechanism we can verify directly. Whether its strength improves generalization is a separate empirical question: too much decay can also make a model underfit.


In [ ]:
weight_decay_run = fit('Weight decay only', weight_decay=0.1)
plot_runs([baseline_run, weight_decay_run], 'Change only weight decay')
show_table(['Selected model', 'L2 parameter norm'], [
    [run['name'], f'{math.sqrt(sum(p.detach().square().sum().item() for p in run["model"].parameters())):.2f}']
    for run in [baseline_run, weight_decay_run]
])


#### Dropout

Add `Dropout(p=0.5)` in the dense head. Random masks are active during training and disabled by `eval()`. The clean training curves below therefore measure the same deterministic prediction rule as validation.

First apply Dropout to ones so that the masks and scaling are visible. **Predict:** what values should appear during training, and what changes in evaluation mode?


In [ ]:
# These CPU tensors are independent of the CNN. Preserve the training RNG state.
with torch.random.fork_rng(devices=[]):
    dropout_demo_x = torch.ones(3, 5)
    dropout_demo_layer = nn.Dropout(p=0.5)
    dropout_demo_layer.train()
    dropout_demo_masks = []
    for dropout_demo_pass in range(3):
        dropout_demo_output = dropout_demo_layer(dropout_demo_x)
        dropout_demo_masks.append(dropout_demo_output)
        print(f'Training pass {dropout_demo_pass + 1}:\n{dropout_demo_output}')
    dropout_demo_layer.eval()
    dropout_demo_eval = dropout_demo_layer(dropout_demo_x)
    print('Evaluation output:\n', dropout_demo_eval)

# Evaluation mode changes the layer's behavior; autograd is controlled separately.
dropout_demo_probe = dropout_demo_x.clone().requires_grad_()
dropout_demo_loss = dropout_demo_layer(dropout_demo_probe).square().sum()
print('Loss in eval() still tracks gradients:', dropout_demo_loss.requires_grad)
with torch.no_grad():
    dropout_demo_no_grad = dropout_demo_layer(dropout_demo_probe).square().sum()
print('Loss with no_grad() tracks gradients:', dropout_demo_no_grad.requires_grad)


**Observe:** Each training pass gives a new mask. Surviving values are scaled by $1/(1-p)$, so at $p=0.5$ they become 2. This preserves the expected activation; one particular mask need not preserve its sample mean. During evaluation, Dropout returns the input unchanged. `eval()` and disabling gradients are separate operations.

The masks demonstrate how Dropout works. Its effect on validation performance depends on the network, dropout probability and training budget. Here `p=0.5` is one illustrative setting, and the next experiment measures its consequences.


In [ ]:
dropout_run = fit('Dropout only', dropout=0.5)
plot_runs([baseline_run, dropout_run], 'Change only dropout')
regularizer_runs = [augmentation_run, weight_decay_run, dropout_run]
show_table(['Run', 'Selected epoch', 'Train loss', 'Val loss', 'Val accuracy'], [
    [run['name'], run['best_epoch'], f'{run["selected"]["train_loss"]:.3f}',
     f'{run["selected"]["val_loss"]:.3f}', f'{run["selected"]["val_acc"]:.1%}']
    for run in [baseline_run, *regularizer_runs]
])


#### Early Stopping

Checkpoint selection retains the best validation loss. Early stopping additionally avoids spending updates after a chosen number of non-improving epochs. It cannot see future epochs: its selected checkpoint may differ from the best over a full run.

Replay the baseline history to see what patience 5 **would have selected**, without another training run. This is a visualization; the repeated fits in section 6 will actually stop and restore their checkpoints.


In [ ]:
def replay_early_stopping(losses, patience):
    best, best_epoch, waiting = float('inf'), 0, 0
    for epoch, loss in enumerate(losses, 1):
        if loss < best:
            best, best_epoch, waiting = loss, epoch, 0
        else:
            waiting += 1
        if waiting >= patience:
            return best_epoch, epoch
    return best_epoch, len(losses)


early_epoch, stop_epoch = replay_early_stopping(baseline_run['history']['val_loss'], PATIENCE)
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, metric, ylabel in zip(axes, ('loss', 'acc'), ('Cross-entropy', 'Accuracy')):
    ax.plot(range(1, MAX_EPOCHS + 1), baseline_run['history'][f'train_{metric}'], label='Clean train')
    ax.plot(range(1, MAX_EPOCHS + 1), baseline_run['history'][f'val_{metric}'], linestyle='--', label='Validation')
    ax.axvline(early_epoch, color='tab:green', linestyle=':', label=f'Selected: {early_epoch}')
    ax.axvline(stop_epoch, color='tab:red', linestyle=':', label=f'Stop: {stop_epoch}')
    ax.axvspan(stop_epoch, MAX_EPOCHS, color='gray', alpha=0.1)
    ax.set(xlabel='Epoch', ylabel=ylabel)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
print(f'Patience {PATIENCE}: select epoch {early_epoch}, stop at {stop_epoch}; '
      f'avoid {MAX_EPOCHS - stop_epoch} later epochs in this replay.')


**Questions:** Which intervention improved validation loss? Which improved accuracy? Did any intervention simply make training harder? Why should we compare validation performance rather than reward the smallest training/validation gap?

These comparisons use one seed. We will check training variability next. Keep the test results hidden while making these choices.

**Read the evidence at three levels:**

* **Mechanism:** do normalization, masks and shrinkage behave as expected?
* **Optimization:** does training loss improve at a matched epoch budget?
* **Generalization:** does validation performance improve, does the change repeat across seeds, and what does the locked test evaluation show?

A modest absolute score can coexist with a useful validation change. An unsuccessful setting is also informative: check its strength, optimization budget and assumptions. These few settings do not establish a general ranking of regularization methods.


### 6. Tune and Verify

#### A Small Random Search

Choose the best of the three regularization families by validation loss. Keep its augmentation/dropout setting fixed and search learning rate and weight decay. Draw six pairs uniformly in **log space**: learning rate $[10^{-4},3\times10^{-3}]$ and weight decay $[10^{-4},3\times10^{-1}]$.

Every search trial has the same eight-epoch budget and seed. Include the chosen family's **first eight epochs** as a reference, reusing its existing history. Select by minimum validation loss within these eight epochs. This makes the comparison use equal budgets; a short search can still favor settings that learn quickly over settings that need more epochs. Zero weight decay is a separate reference value, since $\log 0$ is undefined.

The reference is allowed to win the search. Six short trials do not guarantee an improvement, and the short budget can favor fast initial progress.


In [ ]:
family_run = min(regularizer_runs, key=lambda run: run['selected']['val_loss'])
family_config = family_run['config'].copy()
print('Family selected on validation:', family_run['name'])
search_rng = np.random.default_rng(2026)
search_results = []
for trial in range(N_TRIALS):
    trial_config = {**family_config,
                    'lr': float(10 ** search_rng.uniform(-4, math.log10(3e-3))),
                    'weight_decay': float(10 ** search_rng.uniform(-4, math.log10(0.3)))}
    result = fit(f'Search {trial + 1}/{N_TRIALS}', epochs=SEARCH_EPOCHS, **trial_config)
    search_results.append(result)

reference_losses = family_run['history']['val_loss'][:SEARCH_EPOCHS]
reference_epoch = int(np.argmin(reference_losses)) + 1
reference_result = dict(name='Reference (reused)', config=family_config,
                        best_epoch=reference_epoch,
                        selected={key: values[reference_epoch - 1]
                                  for key, values in family_run['history'].items()})
search_candidates = [reference_result, *search_results]
search_winner = min(search_candidates, key=lambda run: run['selected']['val_loss'])
tuned_config = search_winner['config'].copy()
show_table(['Candidate', 'Learning rate', 'Weight decay', 'Selected epoch', 'Val loss', 'Val accuracy'], [
    [run['name'], f'{run["config"]["lr"]:.2g}', f'{run["config"]["weight_decay"]:.2g}',
     run['best_epoch'], f'{run["selected"]["val_loss"]:.3f}', f'{run["selected"]["val_acc"]:.1%}']
    for run in search_candidates
])
print('Selected configuration:', tuned_config)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.5))
points = ax.scatter([r['config']['lr'] for r in search_results],
                    [r['config']['weight_decay'] for r in search_results],
                    c=[r['selected']['val_loss'] for r in search_results],
                    cmap='viridis_r', s=130, edgecolors='black')
for index, result in enumerate(search_results, 1):
    ax.annotate(str(index), (result['config']['lr'], result['config']['weight_decay']),
                xytext=(6, 5), textcoords='offset points')
ax.set(xscale='log', yscale='log', xlabel='Learning rate', ylabel='Weight decay',
       title='Six random trials; labels identify trial numbers')
fig.colorbar(points, ax=ax, label='Best validation cross-entropy (eight-epoch budget)')
plt.tight_layout()
plt.show()


#### Three Seeds per Final Configuration

Compare the unregularized configuration and the tuned candidate with **the same maximum 30 epochs and patience 5**. Each is trained from scratch using seeds 7, 17 and 27. The split stays fixed: these repeats measure training randomness, not uncertainty from different data splits. The earlier full-length demonstrations used no stopping, so we refit both configurations with the actual stopping rule here. A full-budget demonstration can improve again after a plateau that would already have triggered early stopping. Consequently, even the same-seed refit can select an earlier checkpoint with a different score: it ends before that later improvement.

Choose the final configuration by **mean selected validation loss** across these three seeds, using the unregularized baseline as the tie-breaker. Report accuracy as mean ± sample standard deviation. The test model uses **seed 7, fixed in advance**, from the chosen configuration. We do not pick the luckiest seed, train on validation, or ensemble predictions.


In [ ]:
final_configs = {'Unregularized': BASE_CONFIG.copy(), 'Tuned': tuned_config}
repeated_runs = {}
for name, config in final_configs.items():
    repeated_runs[name] = [fit(f'{name}, seed {seed}', seed=seed, patience=PATIENCE, **config)
                           for seed in FINAL_SEEDS]

seed_statistics = {}
for name, runs in repeated_runs.items():
    accuracies = np.array([r['selected']['val_acc'] for r in runs])
    losses = np.array([r['selected']['val_loss'] for r in runs])
    seed_statistics[name] = dict(mean_acc=float(accuracies.mean()),
                                 std_acc=float(accuracies.std(ddof=1)),
                                 mean_loss=float(losses.mean()), std_loss=float(losses.std(ddof=1)))
chosen_name = min(final_configs, key=lambda name: seed_statistics[name]['mean_loss'])
show_table(['Configuration', 'Seeds', 'Validation loss: mean ± std',
            'Validation accuracy: mean ± std', 'Selected epochs', 'Epochs actually run'], [
    [name, ', '.join(map(str, FINAL_SEEDS)),
     f'{seed_statistics[name]["mean_loss"]:.3f} ± {seed_statistics[name]["std_loss"]:.3f}',
     f'{seed_statistics[name]["mean_acc"]:.1%} ± {seed_statistics[name]["std_acc"]:.1%}',
     ', '.join(str(r['best_epoch']) for r in runs),
     ', '.join(str(r['epochs_run']) for r in runs)]
    for name, runs in repeated_runs.items()
])
print('Final configuration, chosen without test results:', chosen_name)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
for index, (name, runs) in enumerate(repeated_runs.items()):
    values = [r['selected']['val_acc'] for r in runs]
    ax.scatter([index - 0.07, index, index + 0.07], values, label=f'{name}: individual seeds')
    stats = seed_statistics[name]
    ax.errorbar(index, stats['mean_acc'], yerr=stats['std_acc'], fmt='ks', capsize=6)
ax.set(xticks=range(len(final_configs)), xticklabels=list(final_configs),
       ylabel='Selected validation accuracy', title='Training variability on one fixed split')
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


#### Lock the Protocol, Then Evaluate the Test Set

All configuration and checkpoint choices are now complete. The 2,000-image holdout excludes every official-test image used in the previous small version. Its scores have not entered the current search or configuration selection. The final test evaluation compares the **majority-class predictor**, the **seed-7 unregularized CNN**, and the **seed-7 model from the chosen configuration**. If the unregularized configuration wins validation, it remains the final model; improvement is not forced.

The **Wilson 95% interval** describes uncertainty in accuracy for a fixed fitted model. It does not include variability from retraining, split selection or hyperparameter search. Our test sample is balanced by class; use the pooled Wilson interval as an approximate summary for this balanced evaluation population, rather than a statement about other class frequencies or deployment distributions.

McNemar compares paired correct/incorrect outcomes on the **same** test images. Only discordant cases matter: baseline correct/final wrong ($b$) and baseline wrong/final correct ($c$). Under equal marginal correctness, conditional on $b+c$, either direction has probability $1/2$. We use the exact two-sided binomial test. The majority-versus-final comparison checks the trivial baseline; the trained-baseline comparison asks the more demanding question.

**Before executing:** state the final configuration and predict whether a modest accuracy difference will be convincing. After seeing test results, report them; further tuning would need a new independent evaluation.


In [ ]:
def wilson_interval(correct, total, z=1.959963984540054):
    p = correct / total
    denominator = 1 + z * z / total
    center = (p + z * z / (2 * total)) / denominator
    radius = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / denominator
    return max(0.0, center - radius), min(1.0, center + radius)


def exact_mcnemar(baseline_predictions, final_predictions, labels):
    baseline_correct = baseline_predictions == labels
    final_correct = final_predictions == labels
    table = np.array([
        [np.sum(baseline_correct & final_correct), np.sum(baseline_correct & ~final_correct)],
        [np.sum(~baseline_correct & final_correct), np.sum(~baseline_correct & ~final_correct)],
    ], dtype=int)
    b, c = int(table[0, 1]), int(table[1, 0])
    n = b + c
    # Integer arithmetic avoids underflow while adding the binomial lower tail.
    tail_numerator = sum(math.comb(n, k) for k in range(min(b, c) + 1))
    pvalue = min(1.0, 2 * tail_numerator / (2 ** n)) if n else 1.0
    return table, pvalue


In [ ]:
# This is the only test-prediction stage. Both CNNs were fixed using validation.
test_start = time.perf_counter()
locked_baseline = repeated_runs['Unregularized'][0]
locked_final = repeated_runs[chosen_name][0]
assert locked_baseline['seed'] == locked_final['seed'] == FINAL_SEEDS[0] == 7
test_loader = DataLoader(CIFARSubset(test_images, test_labels, plain_transform),
                         batch_size=128, shuffle=False, num_workers=0)
test_results = {}
baseline_model = locked_baseline['model'].to(device)
test_results['Unregularized'] = evaluate(baseline_model, test_loader, predictions=True)
baseline_model.cpu()
if chosen_name == 'Unregularized':
    test_results['Final'] = test_results['Unregularized']
else:
    final_model = locked_final['model'].to(device)
    test_results['Final'] = evaluate(final_model, test_loader, predictions=True)
    final_model.cpu()
majority_predictions = np.full_like(test_labels, majority_class)
test_results['Majority'] = dict(acc=float(np.mean(majority_predictions == test_labels)),
                                predictions=majority_predictions)
for result in test_results.values():
    result['correct'] = int(np.sum(result['predictions'] == test_labels))
    result['interval'] = wilson_interval(result['correct'], len(test_labels))
test_seconds = time.perf_counter() - test_start
show_table(['Predictor', 'Correct / test size', 'Test accuracy', 'Wilson 95% interval'], [
    [name, f'{result["correct"]} / {len(test_labels)}', f'{result["acc"]:.1%}',
     f'[{result["interval"][0]:.1%}, {result["interval"][1]:.1%}]']
    for name, result in test_results.items()
])
print(f'Final configuration: {chosen_name}; fixed seed: 7; test stage: {test_seconds:.2f} s')


Read each paired table with **rows = baseline correct / wrong** and **columns = final correct / wrong**. The top-right cell is $b$; the bottom-left is $c$.

A small p-value is evidence against equal correctness probabilities for these fixed models on this evaluation population. It is not the probability that the final model is better, and it does not establish superiority across datasets or retraining seeds. The two comparisons are reported separately, without a multiple-comparison claim.


In [ ]:
paired_results = {}
for name in ('Majority', 'Unregularized'):
    table, pvalue = exact_mcnemar(test_results[name]['predictions'],
                                 test_results['Final']['predictions'], test_labels)
    paired_results[name] = dict(table=table.tolist(), pvalue=pvalue)
    print(f'{name} versus final:')
    print(table)
    print(f'  b={table[0, 1]}, c={table[1, 0]}; exact two-sided McNemar p={pvalue:.5g}')
    if name == 'Unregularized' and chosen_name == 'Unregularized':
        print('  The baseline was selected as final; predictions are identical.')
delta = test_results['Final']['acc'] - test_results['Unregularized']['acc']
print(f'Final minus unregularized test accuracy: {100 * delta:+.1f} percentage points.')


**Questions:** Do the validation-selected configuration and test results tell a consistent story? How large is the accuracy difference, and how many discordant predictions support it? Why do the three-seed standard deviation and the Wilson interval answer different questions?

A validation-selected model can have lower test accuracy than the baseline. Loss and accuracy can also rank two models differently. Keep the original selection rule and report both; do not choose a replacement using these test results.

**Interpret the scope of the evidence:** The final accuracy describes a simple CNN trained from scratch on 300 images per class in standard mode (100 in quick mode) under this budget. It is not a CIFAR-10 benchmark result, and a small or reversed test difference does not establish that a technique is ineffective in general. Equally, limited data does not excuse a failed comparison: report the observed effect and its uncertainty. Increasing the default training size does not by itself establish a causal data-size effect: this version also expands validation and replaces the test sample, and the same number of epochs gives more optimizer updates. To investigate the data limitation, design a training-size learning curve using one fixed validation protocol and sufficient optimization for each size; keep the current test results out of subsequent tuning.

Use both reference points: outperforming the 10% majority baseline shows useful predictive signal, while the remaining errors show substantial limitations. Neither a modest score nor one unsuccessful regularizer setting settles whether these methods are useful more broadly.


#### Final Experiment Summary

Training/validation entries below use the selected checkpoint unless marked **last epoch**. The one-batch check uses only its 20 examples. Seed rows report means and sample standard deviations on validation; test rows use the seed fixed before testing. Each time is the actual cost of that experiment; reused histories add no training cost.


In [ ]:
notebook_summary = []


def add_summary(step, experiment, result, seconds=None):
    notebook_summary.append(dict(step=step, experiment=experiment, result=result, seconds=seconds))


def describe_run(run):
    s = run['selected']
    return (f'loss train/val {s["train_loss"]:.3f}/{s["val_loss"]:.3f}; '
            f'accuracy {s["train_acc"]:.1%}/{s["val_acc"]:.1%}; epoch {run["best_epoch"]}')


add_summary('1', 'Data audit',
            f'{len(pool_images) + len(test_images):,} distinct archived images; '
            'within/cross-split exact duplicates: 0; previous test images excluded')
add_summary('2', 'Evaluation fixed',
            f'{RUN_MODE}: {len(train_labels):,} / {len(val_labels):,} / {len(test_labels):,} train/val/test; '
            f'majority val {majority_val_accuracy:.1%}; initial loss {initial_loss:.3f}')
add_summary('3', 'Overfit one batch',
            f'20 examples; {mini_steps[-1]} updates; loss {mini_losses[-1]:.6f}; accuracy {mini_accuracies[-1]:.1%}', mini_seconds)
add_summary('4', 'Unregularized: last epoch',
            f'loss train/val {last["train_loss"]:.3f}/{last["val_loss"]:.3f}; accuracy {last["train_acc"]:.1%}/{last["val_acc"]:.1%}')
for run in [baseline_run, *diagnostic_runs]:
    add_summary('4', run['name'], describe_run(run), run['seconds'])
for run in regularizer_runs:
    add_summary('5', run['name'], describe_run(run), run['seconds'])
add_summary('5', 'Early stopping replay',
            f'patience {PATIENCE}; selected epoch {early_epoch}; stop {stop_epoch}; {MAX_EPOCHS - stop_epoch} later epochs avoided')
add_summary('6', 'Random search',
            f'{N_TRIALS} trials × {SEARCH_EPOCHS} epochs; winner {search_winner["name"]}; '
            f'lr={tuned_config["lr"]:.2g}, wd={tuned_config["weight_decay"]:.2g}',
            sum(r['seconds'] for r in search_results))
for name, stats in seed_statistics.items():
    add_summary('6', f'{name}: 3 seeds',
                f'val loss {stats["mean_loss"]:.3f} ± {stats["std_loss"]:.3f}; '
                f'val accuracy {stats["mean_acc"]:.1%} ± {stats["std_acc"]:.1%}',
                sum(r['seconds'] for r in repeated_runs[name]))
for name in ('Majority', 'Unregularized', 'Final'):
    result = test_results[name]
    low, high = result['interval']
    add_summary('6', f'{name}: locked test',
                f'{result["acc"]:.1%}; Wilson 95% [{low:.1%}, {high:.1%}]')
for name, result in paired_results.items():
    add_summary('6', f'McNemar: {name} vs final', f'p={result["pvalue"]:.5g}; paired table {result["table"]}')
show_table(['Step', 'Experiment', 'Result', 'Fit time (s)'], [
    [r['step'], r['experiment'], r['result'], '—' if r['seconds'] is None else f'{r["seconds"]:.1f}']
    for r in notebook_summary
])
TOTAL_SECONDS = time.perf_counter() - NOTEBOOK_START
print(f'Total measured notebook runtime after imports: {TOTAL_SECONDS:.1f} s ({TOTAL_SECONDS / 60:.2f} min) on {device}.')
print(f'Locked test-prediction stage: {test_seconds:.2f} s. No test result was used to choose a setting.')


**Takeaways**

* Interpret this deliberately small experiment within its data, model and compute limits.
* Inspect data and lock the evaluation before tuning.
* Use one-batch memorization to debug the training pipeline.
* Diagnose optimization and capacity using training performance; use validation to judge generalization.
* Verify each mechanism, then measure whether its setting helps validation; benefits are conditional.
* Change one regularizer at a time and keep measurement conditions comparable.
* Repeat promising configurations: the best single run can be misleading.
* Freeze decisions before testing, and report effect sizes and uncertainty alongside accuracy.

**Try it:** for a shorter run, choose `RUN_MODE = 'quick'` before running from the first cell. For a validation-only extension, change one learning-rate range, regularizer or patience value and rerun through the **validation** stage. Explain your choice using those curves. Treat the displayed test results as the final report of the original experiment; tuning after inspecting them would require new independent test data.

**References and data provenance**

* [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html), by Alex Krizhevsky, Vinod Nair and Geoffrey Hinton. The accompanying [subset notes](../assets/datasets/cifar10_training_recipe_subset.md) describe selection, checksums and original indices.
* [PyTorch reproducibility](https://docs.pytorch.org/docs/stable/notes/randomness.html) and [AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html).
* [Wilson intervals in statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportion_confint.html) and [exact McNemar test](https://www.statsmodels.org/stable/generated/statsmodels.stats.contingency_tables.mcnemar.html). The small helpers above use Python's standard library, so SciPy/statsmodels are not required.

The optional [export script](../assets/datasets/prepare_training_recipe_subset.py) regenerates the archive from the full original dataset. It is separate from normal lecture execution, so students do not wait for a full download.

Published studies evaluate these techniques under broader conditions: [Dropout (Srivastava et al., 2014)](https://jmlr.org/papers/v15/srivastava14a.html), [Batch Normalization (Ioffe and Szegedy, 2015)](https://proceedings.mlr.press/v37/ioffe15.html), and [decoupled weight decay (Loshchilov and Hutter, 2019)](https://arxiv.org/abs/1711.05101). These studies report benefits in their evaluated settings; they do not guarantee a gain for every configuration.
